# Medical slice training (Colab T4)
Runtime > Change runtime type > T4 GPU. Run the cells in order. Each step commits and pushes its results,
so your GitHub history shows the work as it happens. Replace the values in the first cell.

In [ ]:
REPO = "https://github.com/YOUR_USER/YOUR_REPO.git"   # your team repo
BRANCH = "medical-training"                          # your own branch
TOKEN = ""   # a GitHub personal access token with repo scope (paste it here, never commit this notebook with it filled in)

In [ ]:
import os
host = REPO.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO
!git clone {host} repo
%cd repo
!git config user.name "Aadya Sharma"
!git config user.email "asharm41@vols.utk.edu"
!git checkout -B {BRANCH}
%cd finetune
!pip install -q -U "transformers>=4.51" peft accelerate faker pytest
!python -m pytest tests -q

In [ ]:
import subprocess

def save(msg):
    """Commit the results so far and push, so progress shows up on GitHub as it happens."""
    run = lambda *a: subprocess.run(a, check=False)
    run("git", "add", "results", "../PLAN.md")
    run("git", "commit", "-q", "-m", msg)
    run("git", "push", "-q", "origin", BRANCH)


## 1. Baselines (no GPU)

In [ ]:
!python evaluate.py --baseline regex --split test --domain medical
save("Medical baseline: regex scores on test")

## 2. DistilBERT token classifier

In [ ]:
!python train_token.py --base distilbert-base-uncased --smoke

In [ ]:
!python train_token.py --base distilbert-base-uncased --out out/b-tok
!python evaluate.py --token-model out/b-tok --split val --domain medical --name b-tok-val-medical
save("Medical: DistilBERT token classifier validation results")

## 3. Qwen3-0.6B with LoRA

In [ ]:
!python evaluate.py --base Qwen/Qwen3-0.6B --split val --domain medical --limit 100 --name zs-qwen3-06b-val-medical
save("Medical: Qwen3-0.6B zero-shot validation results")
!python train_lora.py --base Qwen/Qwen3-0.6B --out out/b --smoke

In [ ]:
!python train_lora.py --base Qwen/Qwen3-0.6B --out out/b
!python evaluate.py --base Qwen/Qwen3-0.6B --adapter out/b --split val --domain medical --name b-val-medical
save("Medical: Qwen3-0.6B LoRA validation results")

## 4. Test set: run ONCE, only after the team picks the final models on validation
Do not tune anything after seeing these numbers.

In [ ]:
!python evaluate.py --token-model out/b-tok --split test --domain medical --name b-tok-test-medical
!python evaluate.py --base Qwen/Qwen3-0.6B --adapter out/b --split test --domain medical --name b-test-medical
!python evaluate.py --token-model out/b-tok --split handwritten --domain medical --name b-tok-hand-medical
!python evaluate.py --base Qwen/Qwen3-0.6B --adapter out/b --split handwritten --domain medical --name b-hand-medical
save("Medical: test and handwritten results for both models")

Read `results/*/errors.jsonl`, write the top 3 failure patterns in your notes, and open a pull request.